# RAVE Dissertation Experiments
**Mihir Apte — MSc Data Science, TCD**

Run cells top to bottom. Select **Runtime > Change runtime type > A100 GPU** first.

**Methods:** Baseline (random) | Semantic v1 (greedy NN) | Semantic v2 (K-means) | Multi-ControlNet (depth+canny) + FreeU
**Videos:** truck, shanghai, street, dog — 4 methods x 4 videos = 16 experiments

Expected total runtime on A100: ~5-6 hours. This notebook is **resume-safe**: if Colab
disconnects partway through Cell 9, just re-run Cell 9 — it automatically skips any
experiment that already produced a GIF and continues with what's left.

In [ ]:
# Cell 1 - Check GPU
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("NO GPU - change runtime type to A100 before continuing")


In [ ]:
# Cell 2 - Clone repo (or pull latest, discarding any local patch changes from a previous run)
import os
REPO = "/content/dissertation-mihir"
if os.path.exists(REPO):
    print("Repo exists - resetting local changes and pulling latest...")
    !cd {REPO} && git checkout -- . && git clean -fd annotator utils pipelines scripts configs 2>/dev/null; cd {REPO} && git pull origin main
else:
    !git clone https://github.com/MihirApte/dissertation-mihir.git {REPO}
os.chdir(REPO)
print("Done. CWD:", os.getcwd())


In [ ]:
# Cell 3 - Install packages
# diffusers is pinned to 0.39.0 - this is the version confirmed to work with a
# modern huggingface_hub. Do NOT let this float to "latest" - untested newer
# diffusers releases have broken this pipeline's API before.
# basicsr, timm, scikit-image are required by the annotator modules that get
# imported at module load time (lineart/hed/zoe/midas/leres) even though we only
# use depth_zoe + canny - without them you get an ImportError before anything runs.
!pip install -q "diffusers==0.39.0" transformers accelerate omegaconf einops scikit-learn \
    scikit-image basicsr timm safetensors \
    Pillow opencv-python imageio imageio-ffmpeg
!pip install -q -U yt-dlp
!pip install -q git+https://github.com/openai/CLIP.git
print("All packages installed.")

import diffusers, transformers
print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)


In [ ]:
# Cell 4 - Patch basicsr (torchvision 0.17+ removed functional_tensor)
import glob
matches = glob.glob("/usr/local/lib/python3.*/dist-packages/basicsr/data/degradations.py")
if matches:
    with open(matches[0]) as f: content = f.read()
    old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    new = "from torchvision.transforms.functional import rgb_to_grayscale"
    if old in content:
        with open(matches[0], "w") as f: f.write(content.replace(old, new))
        print("basicsr patched")
    else:
        print("basicsr already patched (or uses a different import already)")
else:
    print("basicsr not found - skipping (only needed if you hit an ImportError later)")


In [ ]:
# Cell 5 - Patch ZoeDepth
# Fix 1: use GPU (not CPU)
# Fix 2: strict=False to handle newer timm key mismatches
zoe_path = "/content/dissertation-mihir/annotator/zoe/__init__.py"
with open(zoe_path) as f: content = f.read()

# Fix 1: revert any CPU patching
content = content.replace('self.model.to("cpu")', "self.model.to(self.device)")
content = content.replace('torch.from_numpy(image_depth).float().to("cpu")',
                          "torch.from_numpy(image_depth).float().to(self.device)")

# Fix 2: add strict=False to load_state_dict
old = "model.load_state_dict(torch.load(modelpath, map_location=model.device)['model'])"
new = "model.load_state_dict(torch.load(modelpath, map_location=model.device)['model'], strict=False)"
if old in content:
    content = content.replace(old, new)
    print("ZoeDepth strict=False patch applied")
else:
    print("ZoeDepth strict=False already patched")

with open(zoe_path, "w") as f: f.write(content)
print("ZoeDepth patched: GPU mode + strict=False")


In [ ]:
# Cell 6 - Upload truck.mp4 from your computer
# You only need to upload truck.mp4 here.
# The other 3 videos (shanghai, street, dog) are downloaded automatically in Cell 7,
# with a manual-upload fallback if YouTube blocks the download.
import os, shutil
from google.colab import files

VIDEO_DIR = "/content/dissertation-mihir/data/mp4_videos"
os.makedirs(VIDEO_DIR, exist_ok=True)

truck_path = f"{VIDEO_DIR}/truck.mp4"
if os.path.exists(truck_path):
    size = os.path.getsize(truck_path) / 1e6
    print(f"truck.mp4 already present ({size:.1f} MB) - skipping upload")
else:
    print("Select truck.mp4 from your computer...")
    uploaded = files.upload()
    for fname in uploaded:
        shutil.move(fname, truck_path)
        print(f"Saved truck.mp4 ({os.path.getsize(truck_path)/1e6:.1f} MB)")


In [ ]:
# Cell 7 - Download shanghai, street, dog from YouTube (first 10 seconds each)
# Tries two yt-dlp client strategies (YouTube's bot-check sometimes blocks the
# default client on datacenter IPs like Colab's). If both fail, falls back to
# asking you to upload the file manually so the pipeline is never blocked.
import os, subprocess
from google.colab import files

VIDEO_DIR = "/content/dissertation-mihir/data/mp4_videos"
os.makedirs(VIDEO_DIR, exist_ok=True)

videos = {
    "shanghai.mp4": "https://youtu.be/n5cW4FpGvhI",
    "street.mp4":   "https://youtu.be/1XIOmKGjgho",
    "dog.mp4":      "https://youtu.be/7LycCv0PIBo",
}

def try_download(url, out):
    for client in ["android", "web"]:
        cmd = [
            "yt-dlp", "--download-sections", "*0:00-0:10",
            "-f", "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]",
            "--merge-output-format", "mp4",
            "--extractor-args", f"youtube:player_client={client}",
            "-o", out, url,
        ]
        print(f"  trying client={client} ...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        if os.path.exists(out) and os.path.getsize(out) > 0:
            return True
        print(result.stderr[-1500:])
    return False

for filename, url in videos.items():
    out = f"{VIDEO_DIR}/{filename}"
    if os.path.exists(out) and os.path.getsize(out) > 0:
        print(f"{filename} already exists - skipping")
        continue
    print(f"Downloading {filename}...")
    ok = try_download(url, out)
    if not ok:
        print(f"Automatic download failed for {filename}.")
        print(f"Please download {url} yourself (e.g. via a browser) and upload it below.")
        uploaded = files.upload()
        for fname in uploaded:
            os.rename(fname, out)
            print(f"Saved {filename} from manual upload.")

# Verify all 4
print("\nVideo check:")
for v in ["truck.mp4", "shanghai.mp4", "street.mp4", "dog.mp4"]:
    path = f"{VIDEO_DIR}/{v}"
    if os.path.exists(path):
        print(f"  OK  {v}  ({os.path.getsize(path)/1e6:.1f} MB)")
    else:
        print(f"  MISSING  {v}")


In [ ]:
# Cell 8 - Define the resumable experiment runner
# 16 experiments = 4 methods x 4 videos. Each one is checked against results/
# before running, so re-running this cell after a disconnect skips anything
# already finished and only does the remaining work.
#
# Output streams live below (including tqdm progress bars), and a heartbeat
# message prints every 30s of silence so long quiet stretches (model download,
# DDIM inversion setup before the progress bar appears) don't look like a freeze.
import os, glob, subprocess, sys, time, threading, yaml

os.chdir("/content/dissertation-mihir")

EXPERIMENTS = [
    ("configs/baseline_random.yaml",       "Truck - Baseline (random)"),
    ("configs/semantic_shuffle.yaml",      "Truck - Semantic v1 (greedy NN)"),
    ("configs/truck_kmeans.yaml",          "Truck - Semantic v2 (K-means)"),
    ("configs/truck_multicontrol.yaml",    "Truck - Multi-ControlNet + FreeU"),

    ("configs/shanghai_baseline.yaml",     "Shanghai - Baseline (random)"),
    ("configs/shanghai_semantic.yaml",     "Shanghai - Semantic v1 (greedy NN)"),
    ("configs/shanghai_kmeans.yaml",       "Shanghai - Semantic v2 (K-means)"),
    ("configs/shanghai_multicontrol.yaml", "Shanghai - Multi-ControlNet + FreeU"),

    ("configs/street_baseline.yaml",       "Street - Baseline (random)"),
    ("configs/street_semantic.yaml",       "Street - Semantic v1 (greedy NN)"),
    ("configs/street_kmeans.yaml",         "Street - Semantic v2 (K-means)"),
    ("configs/street_multicontrol.yaml",   "Street - Multi-ControlNet + FreeU"),

    ("configs/dog_baseline.yaml",          "Dog - Baseline (random)"),
    ("configs/dog_semantic.yaml",          "Dog - Semantic v1 (greedy NN)"),
    ("configs/dog_kmeans.yaml",            "Dog - Semantic v2 (K-means)"),
    ("configs/dog_multicontrol.yaml",      "Dog - Multi-ControlNet + FreeU"),
]

def already_done(save_folder):
    return len(glob.glob(f"results/*/{save_folder}/*/*.gif")) > 0

def video_missing(config_path):
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    video_path = f"data/mp4_videos/{cfg['video_name']}.mp4"
    return not os.path.exists(video_path)

def stream_run(cmd, heartbeat_secs=30):
    """Run a subprocess, streaming its raw output live (so tqdm progress bars
    actually animate instead of only appearing once a line completes), and
    printing a heartbeat if it goes quiet for a while."""
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    last_output = time.time()
    stop = threading.Event()

    def watchdog():
        while not stop.is_set():
            time.sleep(5)
            idle = time.time() - last_output
            if idle > heartbeat_secs and not stop.is_set():
                print(f"   ... still running, no new output for {int(idle)}s "
                      f"(normal during model download / DDIM inversion setup)", flush=True)

    t = threading.Thread(target=watchdog, daemon=True)
    t.start()
    try:
        while True:
            chunk = process.stdout.read(1024)
            if not chunk:
                break
            sys.stdout.write(chunk.decode(errors="replace"))
            sys.stdout.flush()
            last_output = time.time()
    finally:
        stop.set()
        process.stdout.close()
    return process.wait()

def run_config(config_path, label, idx, total):
    tag = f"[{idx}/{total}]"
    if not os.path.exists(config_path):
        print(f"{tag} [SKIP] {label}: config not found ({config_path})")
        return "skip"
    if video_missing(config_path):
        print(f"{tag} [SKIP] {label}: source video missing - run Cell 6/7 first")
        return "skip"
    with open(config_path) as f:
        save_folder = yaml.safe_load(f)["save_folder"]
    if already_done(save_folder):
        print(f"{tag} [SKIP] {label}: already completed (found existing GIF)")
        return "done"
    print(f"\n{'='*70}\n{tag} [RUN] {label}\n{'='*70}", flush=True)
    t0 = time.time()
    ret = stream_run(["python3", "scripts/run_experiment.py", config_path])
    mins = (time.time() - t0) / 60
    if ret == 0:
        print(f"{tag} [OK] {label} finished in {mins:.1f} min", flush=True)
        return "ok"
    else:
        print(f"{tag} [FAIL] {label} (exit code {ret}) - continuing to next experiment", flush=True)
        return "fail"

print(f"{len(EXPERIMENTS)} experiments queued.\n")


In [ ]:
# Cell 9 - Run ALL experiments (safe to re-run after a disconnect)
# Live output for the currently-running experiment streams below, with a
# progress line after each one so you always know where things stand.
run_start = time.time()
summary = {}
total = len(EXPERIMENTS)
for i, (config_path, label) in enumerate(EXPERIMENTS, 1):
    summary[label] = run_config(config_path, label, i, total)
    done_count = sum(1 for s in summary.values() if s in ("ok", "done"))
    elapsed = (time.time() - run_start) / 60
    print(f"--- progress: {done_count}/{total} complete | {elapsed:.1f} min elapsed since this cell started ---\n", flush=True)

print("=" * 60)
print("SUMMARY")
print("=" * 60)
for label, status in summary.items():
    print(f"  [{status.upper():<5}] {label}")

failed = [l for l, s in summary.items() if s == "fail"]
if failed:
    print("\nSome experiments failed - re-run this cell to retry them (completed ones will be skipped).")


In [ ]:
# Cell 10 - Show full metrics table (Warp Error + CLIP Score, all methods x all videos)
import os, subprocess
os.chdir("/content/dissertation-mihir")
results_file = "results/metrics_all_methods.txt"

print("Computing metrics for all completed experiments...")
ret = subprocess.run(["python3", "compute_metrics_all.py", "--device", "cuda"])
if ret.returncode != 0:
    print("CUDA metrics run failed, retrying on CPU...")
    subprocess.run(["python3", "compute_metrics_all.py", "--device", "cpu"])

if os.path.exists(results_file):
    with open(results_file) as f:
        print(f.read())
else:
    print("metrics_all_methods.txt not found - check the output above for errors.")


In [ ]:
# Cell 11 - Download all results (GIFs + metrics) as a zip
import shutil
from google.colab import files
shutil.make_archive("/content/rave_results", "zip", "/content/dissertation-mihir/results")
files.download("/content/rave_results.zip")
print("Downloading rave_results.zip...")
print("If you get a path-too-long error on Windows extraction, use 7-zip.")
